# Greek Medical Report Anonymization

Run the cells from top to bottom.

If your model is stored in the default Google Drive folder, you should not need to edit anything.


In [ ]:
%cd /content
!rm -rf greek-medical-anonymization
!git clone https://github.com/VanessaLislevand/greek-medical-anonymization.git
%cd /content/greek-medical-anonymization
%pip install -e ".[ml,ui]"


In [ ]:
from pathlib import Path
import shutil

from google.colab import drive

drive.mount('/content/drive')

repo_root = Path('/content/greek-medical-anonymization')
repo_model_dir = repo_root / 'models' / 'xlmr_phi_final'

DEFAULT_DRIVE_MODEL_DIR = Path(
    '/content/drive/MyDrive/Archimedes_Anonymization/exported_models/xlmr_phi_final_all_data'
)

drive_model_dir = DEFAULT_DRIVE_MODEL_DIR
print(f'Default model path: {drive_model_dir}')

if not drive_model_dir.exists():
    custom_path = input(
        'Default model path not found. Paste the Google Drive path to the model folder: '
    ).strip()
    drive_model_dir = Path(custom_path)

if not drive_model_dir.exists():
    raise FileNotFoundError('Model folder not found in Google Drive.')

if repo_model_dir.exists() or repo_model_dir.is_symlink():
    if repo_model_dir.is_symlink() or repo_model_dir.is_file():
        repo_model_dir.unlink()
    else:
        shutil.rmtree(repo_model_dir)

repo_model_dir.symlink_to(drive_model_dir, target_is_directory=True)

print('Model folder linked successfully.')
print('Model files:')
for path in sorted(repo_model_dir.iterdir()):
    if path.is_file():
        print(f'- {path.name}')


In [ ]:
import os
import socket
import subprocess
import sys
import time

from google.colab import output

os.chdir('/content/greek-medical-anonymization')

try:
    streamlit_process
except NameError:
    streamlit_process = None

if streamlit_process is not None and streamlit_process.poll() is None:
    print('The web app is already running.')
else:
    streamlit_process = subprocess.Popen(
        [
            sys.executable,
            '-m',
            'streamlit',
            'run',
            'src/greek_med_anonymizer/web_app.py',
            '--server.port',
            '8501',
            '--server.headless',
            'true',
            '--browser.gatherUsageStats',
            'false',
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

ready = False
for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', 8501), timeout=1):
            ready = True
            break
    except OSError:
        time.sleep(1)

if not ready:
    raise RuntimeError('The Streamlit app did not start in time. Run this cell again.')

output.serve_kernel_port_as_iframe(8501, height='900')
